In [ ]:
# ===== Import Libraries yang Digunakan =====

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import pywt
from scipy.signal import butter, filtfilt, resample
from scipy.stats import zscore
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
# ===== Konfigurasi Parameter yang Digunakan =====

fs = 250     
windowsize = 2500        
normal = 0        
afib = 1            

In [ ]:
# ===== Fungsi untuk Preprocessing Sinyal =====

# 1. Bandpass Filter Sinyal EKG
def apply_bandpass_filter(sig, fs=fs, lowcut=0.5, highcut=40.0, order=3):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, sig)

# 2. Preprocessing untuk Data Training
def process_training_row(row_signal_400hz):
    sig_250hz = resample(row_signal_400hz, windowsize)
    sig_bpf = apply_bandpass_filter(sig_250hz, fs=fs)
    coeffs = pywt.wavedec(sig_bpf, 'db5', level=5)
    coeffs[0] = np.zeros_like(coeffs[0])
    sig_dwt = pywt.waverec(coeffs, 'db5')[:len(sig_bpf)]
    return zscore(sig_dwt)

# 3. Preprocessing untuk Data Testing
def process_testing_column(long_signal_250hz):
    sig_bpf = apply_bandpass_filter(long_signal_250hz, fs=fs)
    coeffs = pywt.wavedec(sig_bpf, 'db5', level=5)
    coeffs[0] = np.zeros_like(coeffs[0])
    sig_dwt = pywt.waverec(coeffs, 'db5')[:len(sig_bpf)]
    sig_norm = zscore(sig_dwt)
    
    jendela_terkumpul = []
    for i in range(0, len(sig_norm) - windowsize + 1, windowsize):
        jendela_terkumpul.append(sig_norm[i : i + windowsize])        
    return np.array(jendela_terkumpul)

In [ ]:
# ===== Konfigurasi Path untuk Dataset, Model, dan Logs =====

traindata = "/Users/alstanlin/Desktop/CAT-NET for Arrhythmia Detection/traindata.csv"
testdata  = "/Users/alstanlin/Desktop/CAT-NET for Arrhythmia Detection/testdata.csv"

project_path = "/Users/alstanlin/Desktop/CAT-NET for Arrhythmia Detection/"
model_path = os.path.join(project_path, "catnet_afib_model.h5")
logdir = os.path.join(project_path, "logs")

In [ ]:
# ===== Persiapan Data Training =====

train_df = pd.read_csv(traindata)
n_sample = 1000

# Labeling harus dilakukan secara berurutan karena data sudah diurutkan (AFib di awal, Normal di akhir)
n_afib = 500    # Row  0 - 499 = AFib (label=1)
n_normal = 500  # Row 500- 999 = Normal (label=0)

X_train_raw = train_df.iloc[0:n_sample, 0:4000].values
y_train_urut = np.concatenate([np.ones(n_afib), np.zeros(n_normal)])
assert len(y_train_urut) == n_sample, f"Label count mismatch: {len(y_train_urut)} != {n_sample}"
print(f"Distribusi Label {np.sum(y_train_urut==1):.0f} AFib, {np.sum(y_train_urut==0):.0f} Normal")

print("[INFO] Memulai preprocessing data training...")
processed_train = [process_training_row(row) for row in X_train_raw]
X_train_urut = np.array(processed_train)
X_train_urut = np.expand_dims(X_train_urut, axis=-1)
acak_indeks = np.arange(X_train_urut.shape[0])
np.random.shuffle(acak_indeks)
x_train = X_train_urut[acak_indeks]
y_train = y_train_urut[acak_indeks]
print(f"[INFO] Data Training Siap (Acak): {x_train.shape}")

In [ ]:
# ===== Augmentasi Data Training =====

def augment_signal(signal, fs=250):
    augmented = []
    
    # 1. Jittering: tambahkan noise gaussian kecil
    noise = np.random.normal(0, 0.05, len(signal))
    augmented.append(signal + noise)
    
    # 2. Scaling: perbesar/kecilkan amplitudo +/-10%
    scale = np.random.uniform(0.9, 1.1)
    augmented.append(signal * scale)
    
    # 3. Time shifting: geser sinyal +/-50 sampel
    shift = np.random.randint(-50, 50)
    augmented.append(np.roll(signal, shift))
    return augmented

X_augmented = list(X_train_urut)
y_augmented = list(y_train_urut)

for i in range(len(X_train_urut)):
    signal = X_train_urut[i].squeeze()
    aug_signals = augment_signal(signal)
    for aug_sig in aug_signals:
        X_augmented.append(aug_sig.reshape(-1, 1))
        y_augmented.append(y_train_urut[i])

X_train_aug = np.array(X_augmented)
y_train_aug = np.array(y_augmented)
print(f"Data sebelum augmentasi: {X_train_urut.shape[0]}")
print(f"Data setelah augmentasi: {X_train_aug.shape[0]}")

# Pengacakan ulang data setelah augmentasi
acak_indeks = np.arange(X_train_aug.shape[0])
np.random.shuffle(acak_indeks)
x_train = X_train_aug[acak_indeks]
y_train = y_train_aug[acak_indeks]

In [ ]:
# ===== Layer Attention untuk Model CAT-NET =====

# 1. Channel Attention Layer
class ChannelAttention(tf.keras.layers.Layer):
    def __init__(self, filters, ratio=8, **kwargs): 
        super(ChannelAttention, self).__init__(**kwargs)
        self.filters = filters
        self.ratio = ratio

    def build(self, input_shape):
        self.shared_layer_one = tf.keras.layers.Dense(self.filters // self.ratio, activation='relu', kernel_initializer='he_normal')
        self.shared_layer_two = tf.keras.layers.Dense(self.filters, kernel_initializer='he_normal')

    def call(self, inputs):
        avg_pool = tf.keras.layers.GlobalAveragePooling1D()(inputs)
        avg_pool = self.shared_layer_one(avg_pool)
        avg_pool = self.shared_layer_two(avg_pool)
        max_pool = tf.keras.layers.GlobalMaxPooling1D()(inputs)
        max_pool = self.shared_layer_one(max_pool)
        max_pool = self.shared_layer_two(max_pool)
        attention = tf.keras.layers.Add()([avg_pool, max_pool])
        attention = tf.keras.layers.Activation('sigmoid')(attention)
        attention = tf.keras.layers.Reshape((1, self.filters))(attention)
        return tf.keras.layers.Multiply()([inputs, attention])
    
# 2. Temporal/Spatial Attention Layer
class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv1d = tf.keras.layers.Conv1D(filters=1, kernel_size=self.kernel_size, padding='same', activation='sigmoid', kernel_initializer='he_normal', use_bias=False)
    
    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=2, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=2, keepdims=True)
        attention = tf.concat([avg_pool, max_pool], axis=2)
        attention = self.conv1d(attention)
        return inputs * attention

In [ ]:
# ===== Layer Transformer Encoder untuk Model CAT-NET =====

class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, num_heads, d_model, dff, dropout_rate, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.multi_head_attention = tf.keras.layers.MultiHeadAttention(key_dim=d_model, num_heads=num_heads)
        self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
        self.layer_norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dense1 = tf.keras.layers.Dense(dff, activation='relu')
        self.dense2 = tf.keras.layers.Dense(d_model)
        self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
        self.layer_norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=True):
        attn_output = self.multi_head_attention(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)
        ffn_output = self.dense1(out1)
        ffn_output = self.dense2(ffn_output)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)

In [ ]:
# ===== Fungsi untuk Positional Encoding pada Model CAT-NET =====

def positional_encoding(seq_len, d, n=10000):
    P = np.zeros((seq_len, d))
    for k in range(seq_len):
        for i in np.arange(int(d/2)):
            denominator = np.power(n, 2*i/d)
            P[k, 2*i] = np.sin(k/denominator)
            P[k, 2*i+1] = np.cos(k/denominator)
    return tf.constant(P, dtype=tf.float32)

In [ ]:
# ===== Fungsi untuk Membangun Arsitektur Model CAT-NET =====

def buildModel():
    inputs = tf.keras.Input(shape=(windowsize, 1))

    # CNN Block 1
    x = tf.keras.layers.Conv1D(16, 21, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = ChannelAttention(16, 8)(x)
    x = SpatialAttention(7)(x)
    x = tf.keras.layers.MaxPool1D(3, strides=2, padding='same')(x)

    # CNN Block 2
    x = tf.keras.layers.Conv1D(32, 23, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = ChannelAttention(32, 8)(x)
    x = SpatialAttention(7)(x)
    x = tf.keras.layers.MaxPool1D(3, strides=2, padding='same')(x)

    # CNN Block 3
    x = tf.keras.layers.Conv1D(64, 25, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = ChannelAttention(64, 8)(x)
    x = SpatialAttention(7)(x)
    x = tf.keras.layers.AvgPool1D(3, strides=2, padding='same')(x)

    # CNN Block 4
    x = tf.keras.layers.Conv1D(128, 27, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)

    # Positional Encoding sebelum masuk ke Transformer
    pos_enc = positional_encoding(seq_len=x.shape[1], d=x.shape[2])
    x = x + pos_enc

    # Transformer Encoder Block
    x = TransformerEncoder(num_heads=4, d_model=128, dff=256, dropout_rate=0.1)(x)

    # Meratakan dimensi keluaran Transformer
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
    # Dense Classifier
    x = tf.keras.layers.Dropout(rate=0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(2, activation='softmax')(x)
    
    return tf.keras.Model(inputs, outputs)

In [ ]:
# ===== Konfigurasi Training Data =====

ratio = 0.2
if os.path.exists(model_path):
    print(f"[INFO] Ditemukan model yang sudah dilatih di: {model_path}")
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            'ChannelAttention': ChannelAttention,
            'SpatialAttention': SpatialAttention,
            'TransformerEncoder': TransformerEncoder
        }
    )
else:
    print("[INFO] Model belum ada. Memulai proses Training...")
    model = buildModel()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(filepath=project_path + "weights.weights.h5", save_weights_only=True, monitor='val_accuracy', mode='max', save_best_only=True)
    early_stop_cb = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True,verbose=1)
    reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6,verbose=1)
    history = model.fit(x_train, y_train, epochs=100, batch_size=32, validation_split=ratio, callbacks=[tensorboard_callback, checkpoint_cb, early_stop_cb, reduce_lr_cb])
    model.save(model_path)
    print(f"[INFO] Training Selesai! Model disimpan di: {model_path}")

In [ ]:
# ===== Tampilkan Ringkasan Arsitektur Model =====

model.summary()

In [ ]:
# ===== Plot Kurva Accuracy dan Loss dari Training =====

# 1. Akurasi
plt.title('Akurasi')
plt.plot(history.history['accuracy'], label='Akurasi Training')
plt.plot(history.history['val_accuracy'], label='Akurasi Testing')
plt.legend()
plt.show()

# 2. Loss
plt.title('Loss')
plt.plot(history.history['loss'], label='Loss Training')
plt.plot(history.history['val_loss'], label='Loss Testing')
plt.legend()
plt.show()

In [ ]:
# ===== Load & Prediksi Data Testing =====

test_df = pd.read_csv(testdata)
test_signal_1d = test_df.iloc[:, 2].dropna().values 
X_test = process_testing_column(test_signal_1d)
X_test = np.expand_dims(X_test, axis=-1)

prediksi_prob = model.predict(X_test)
prediksi_kelas = np.argmax(prediksi_prob, axis=1)

label_map = {0: 'Normal', 1: 'AFib'}
print(f"\n{'Window':<10} {'Prediksi':<10} {'Confidence':<15}")
print("-" * 35)
for i, (kelas, prob) in enumerate(zip(prediksi_kelas, prediksi_prob)):
    confidence = prob[kelas] * 100
    print(f"{i+1:<10} {label_map[kelas]:<10} {confidence:.1f}%")

n_afib = np.sum(prediksi_kelas == 1)
n_normal = np.sum(prediksi_kelas == 0)
print(f"\nTotal: {n_normal} Normal, {n_afib} AFib dari {len(prediksi_kelas)} window")

In [ ]:
# ===== Evaluasi Akurasi dan Loss pada Data Training & Testing =====

import matplotlib.pyplot as plt
import seaborn as sns

train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)

test_loss, test_acc = model.evaluate(X_test, y_test_asli, verbose=0)
print('Training Accuracy : %.2f%%' % (train_acc * 100))
print('Training Loss     : %.2f%%' % (train_loss * 100))
print('Testing Accuracy  : %.2f%%' % (test_acc * 100))
print('Testing Loss      : %.2f%%' % (test_loss * 100))

In [ ]:
# ===== Plot Confusion Matrix dengan Heatmap =====

def plotHeatMap(y_true, y_pred):
    con_mat = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(con_mat, annot=True, square=True, fmt='g', cmap='Greens', xticklabels=['Normal', 'AFib'], yticklabels=['Normal', 'AFib'],cbar=False)
    plt.title('Confusion Matrix', pad=15, fontsize=14)
    plt.xlabel('Predicted Labels', fontsize=12)
    plt.ylabel('True Labels', fontsize=12)
    plt.tight_layout()
    plt.show()
    
plotHeatMap(y_test_eval, y_pred)

In [ ]:
# ===== Evaluasi Model: Accuracy, Precision, Sensitivity (Recall), Specificity, F1-Score =====

y_pred = np.argmax(model.predict(X_test_eval), axis=1)
y_true = y_test_eval

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
accuracy    = accuracy_score(y_true, y_pred)
precision   = precision_score(y_true, y_pred, pos_label=1)
sensitivity = recall_score(y_true, y_pred, pos_label=1)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1          = f1_score(y_true, y_pred, pos_label=1)

df_metrics = pd.DataFrame({'Metrik': ['Accuracy', 'Precision', 'Sensitivity', 'Specificity', 'F1-Score', 'Recall'], 'Nilai': [accuracy, precision, sensitivity, specificity, f1, recall],'Persentase': [f'{accuracy*100:.2f}%', f'{precision*100:.2f}%', f'{sensitivity*100:.2f}%', f'{specificity*100:.2f}%', f'{f1*100:.2f}%', f'{recall*100:.2f}%'], 'Rumus': ['(TP+TN) / (TP+TN+FP+FN)','TP / (TP+FP)','TP / (TP+FN)','TN / (TN+FP)','2×(Prec×Rec) / (Prec+Rec)','= Sensitivity']})
df_metrics.index = df_metrics.index + 1
display(df_metrics)

df_cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=[0, 1]), index=['Aktual Normal', 'Aktual AFib'], columns=['Prediksi Normal', 'Prediksi AFib'])
print(f"\nConfusion Matrix (TP={tp}, TN={tn}, FP={fp}, FN={fn}):")
display(df_cm)